In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# News Headline Classification Using Deep Learning

In this project, five neural network architectures are compared for news headline classification:

1. ANN
2. RNN
3. LSTM
4. BiLSTM
5. GRU

All models use the same dataset, preprocessing, GloVe embeddings, training settings, and evaluation metrics for a fair comparison.

### Pipeline

News Title → Cleaning → Tokenization → Vocabulary → Token IDs → Dynamic Padding → GloVe Embeddings → Model → News Category

## Step 1: Connect Google Drive

Google Drive is mounted so that the dataset and generated results remain permanently saved even if the Colab session disconnects.

The dataset is stored permanently inside the Google Drive folder:

MyDrive/News Classifier using Deep Learning/

In [3]:
FILE = "/content/drive/MyDrive/News Classifier using Deep Learning/corrected_news.csv"

import os

print("File exists:", os.path.exists(FILE))

File exists: True


## Step 3: Import Required Libraries

PyTorch is used for neural networks, Pandas and NumPy for data processing, Scikit-learn for dataset splitting and evaluation, and Gensim for pretrained GloVe embeddings.

In [4]:
!pip install --upgrade gensim
import re
import random
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

import gensim.downloader as api

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from torch.utils.data import Dataset, DataLoader

from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 79.8 MB/s eta 0:00:00


## Step 4: Configuration

Common hyperparameters are kept the same for all models to ensure a fair comparison.

In [5]:
SEED = 42
BATCH = 32
MIN_FREQ = 2

EMB = 100
HID = 128
LAYERS = 2
DROP = 0.3

EPOCHS = 20
PATIENCE = 4

LR = 5e-4
WD = 1e-4

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

LABELS = [
    "Business",
    "Energy",
    "Health",
    "Markets",
    "Politics",
    "Technology"
]

L2I = {
    label: i
    for i, label in enumerate(LABELS)
}

print("Device:", DEVICE)

Device: cuda


## Step 5: Reproducibility

A fixed random seed is used so that the experiment produces more consistent results across runs.

In [6]:
def set_seed():

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

set_seed()

## Step 6: Text Cleaning and Tokenization

Each news headline is:

- converted to lowercase,
- cleaned by removing punctuation and special characters,
- stripped of extra spaces,
- split into individual words.

The standalone token "s" is removed because apostrophes such as "company's" can become "company s" after punctuation removal.

In [7]:
def normalize(text):

    return re.sub(
        r"\s+",
        " ",
        str(text).lower().strip()
    )


def tokenize(text):

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    tokens = text.split()

    return [
        word
        for word in tokens
        if word != "s"
    ]
tokenize("Nvidia's shares rise after strong earnings!")


['nvidia', 'shares', 'rise', 'after', 'strong', 'earnings']

## Step 7: Load the Dataset

Only the news headline (`Title`) and corrected target label (`Final_Category`) are required.

In [8]:
df = pd.read_csv(FILE)

df = df[
    ["Title", "Final_Category"]
].dropna()

df = df[
    df["Final_Category"].isin(L2I)
].copy()

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (4997, 2)


,Title,Final_Category
0,Vietnam Airlines data leak exposes a crisis of...,Technology
1,DOJ seizes $15 billion bitcoin in SE Asia cryp...,Markets
2,Think It's Too Late to Buy IonQ? Here's the 1 ...,Markets
3,Intel's Crucial Panther Lake Chips Start Produ...,Technology
4,3 Robotics Stocks to Buy in October,Markets


## Step 8: Remove Duplicate Headlines

Normalized headlines are used to identify and remove duplicate news titles.

In [9]:
df["Normalized_Title"] = (
    df["Title"].map(normalize)
)

print("Before:", len(df))

df = (
    df
    .drop_duplicates("Normalized_Title")
    .reset_index(drop=True)
)

print("After:", len(df))

Before: 4997
After: 4497


## Step 9: Train, Validation and Test Split

The dataset is divided into:

- 80% Training
- 10% Validation
- 10% Testing

Stratified splitting preserves the class distribution.

In [10]:
train, temp = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["Final_Category"]
)

val, test = train_test_split(
    temp,
    test_size=0.50,
    random_state=SEED,
    stratify=temp["Final_Category"]
)

train = train.copy()
val = val.copy()
test = test.copy()

print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

Train: 3597
Validation: 450
Test: 450


## Step 10: Tokenize Headlines

Every news headline is converted into a list of tokens.

In [11]:
for data in (train, val, test):

    data["tokens"] = (
        data["Title"].map(tokenize)
    )

train = train[
    train["tokens"].map(len) > 0
].copy()

val = val[
    val["tokens"].map(len) > 0
].copy()

test = test[
    test["tokens"].map(len) > 0
].copy()

train[["Title", "tokens"]].head()

,Title,tokens
1766,"Group including Nvidia, BlackRock buying Align...","[group, including, nvidia, blackrock, buying, ..."
2159,ATYR SHAREHOLDER ACTION REMINDER: Faruqi & Far...,"[atyr, shareholder, action, reminder, faruqi, ..."
3461,Defense Firm CSG Sounds Out Investors for €3 B...,"[defense, firm, csg, sounds, out, investors, f..."
1204,âHidden Money In India Is Insaneâ: X User ...,"[hidden, money, in, india, is, insane, x, user..."
29,Coalition torn over super tax fight as budget ...,"[coalition, torn, over, super, tax, fight, as,..."


## Step 11: Vocabulary Creation

The vocabulary is created only from the training dataset.

Special tokens:

- `<PAD>` = 0
- `<UNK>` = 1

Words appearing fewer than two times are mapped to `<UNK>`.

In [12]:
word_counts = Counter(
    word
    for tokens in train["tokens"]
    for word in tokens
)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, count in word_counts.items():

    if count >= MIN_FREQ:

        vocab[word] = len(vocab)

print("Unique training words:", len(word_counts))
print("Vocabulary size:", len(vocab))

Unique training words: 8488
Vocabulary size: 3981


## Step 12: Convert Tokens to Numerical IDs

Neural networks cannot directly process text, so every token is replaced by its vocabulary ID.

In [13]:
def to_ids(tokens):

    return [
        vocab.get(
            word,
            vocab["<UNK>"]
        )
        for word in tokens
    ]


for data in (train, val, test):

    data["ids"] = (
        data["tokens"].map(to_ids)
    )

    data["y"] = (
        data["Final_Category"].map(L2I)
    )

train[
    ["tokens", "ids", "Final_Category", "y"]
].head()

,tokens,ids,Final_Category,y
1766,"[group, including, nvidia, blackrock, buying, ...","[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]",Technology,5
2159,"[atyr, shareholder, action, reminder, faruqi, ...","[16, 17, 18, 19, 20, 20, 21, 22, 23, 24, 25, 2...",Health,2
3461,"[defense, firm, csg, sounds, out, investors, f...","[29, 30, 1, 31, 32, 27, 33, 34, 15, 35, 36, 37...",Markets,3
1204,"[hidden, money, in, india, is, insane, x, user...","[40, 41, 10, 42, 43, 1, 44, 45, 46, 1, 47, 48,...",Markets,3
29,"[coalition, torn, over, super, tax, fight, as,...","[54, 1, 55, 56, 57, 58, 37, 59, 60, 61]",Politics,4


## Step 13: Pretrained GloVe Embeddings

GloVe provides pretrained numerical representations of words.

The `glove-wiki-gigaword-100` model represents each word using a 100-dimensional vector.

Using pretrained embeddings helps the model understand semantic relationships between words before training on our small dataset.

In [ ]:
print("Loading GloVe...")

glove = api.load(
    "glove-wiki-gigaword-100"
)

print("GloVe loaded.")

Loading GloVe...
[==================================================] 100.0% 128.1/128.1MB downloaded
GloVe loaded.


## Step 14: Create the Embedding Matrix

Each word in our vocabulary receives its corresponding pretrained GloVe vector.

Words unavailable in GloVe receive a small random vector, while the padding token receives a zero vector.

In [ ]:

W = np.random.normal(
    0,
    0.05,
    (len(vocab), EMB)
).astype("float32")

W[vocab["<PAD>"]] = 0

found = 0

for word, index in vocab.items():

    if word in glove:

        W[index] = glove[word]
        found += 1

print(
    f"GloVe coverage: {found}/{len(vocab)}"
)

W = torch.tensor(
    W,
    dtype=torch.float32
)

del glove

GloVe coverage: 3784/3981


## Step 15: PyTorch Dataset

A custom PyTorch Dataset returns:

- token IDs as the model input,
- category ID as the target.

In [ ]:
class NewsDataset(Dataset):

    def __init__(self, data):

        self.x = data["ids"].tolist()
        self.y = data["y"].tolist()


    def __len__(self):

        return len(self.y)


    def __getitem__(self, index):

        return (
            torch.tensor(
                self.x[index],
                dtype=torch.long
            ),
            torch.tensor(
                self.y[index],
                dtype=torch.long
            )
        )

## Step 16: Dynamic Padding

News headlines have different lengths.

Dynamic padding makes all headlines within a batch equal in length by adding `<PAD>` tokens only where required.

In [ ]:
def collate(batch):

    sequences, labels = zip(*batch)

    lengths = torch.tensor(
        [
            len(sequence)
            for sequence in sequences
        ]
    )

    padded = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    labels = torch.stack(labels)

    return padded, lengths, labels

## Step 17: DataLoaders

DataLoaders divide the dataset into batches of 32 samples during training and evaluation.

In [ ]:
def make_loader(data, shuffle=False):

    return DataLoader(
        NewsDataset(data),
        batch_size=BATCH,
        shuffle=shuffle,
        collate_fn=collate
    )


train_loader = make_loader(
    train,
    shuffle=True
)

val_loader = make_loader(val)

test_loader = make_loader(test)
x, lengths, y = next(
    iter(train_loader)
)

print("Input shape:", x.shape)
print("Lengths:", lengths.shape)
print("Labels:", y.shape)

NameError: name 'NewsDataset' is not defined